In [0]:
spark


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/faers"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023Q3/,faers_ascii_2023Q3/,0,1771234754073
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023Q3.zip,faers_ascii_2023Q3.zip,63036207,1771226156000
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023Q4/,faers_ascii_2023Q4/,0,1771234754073
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023Q4.zip,faers_ascii_2023Q4.zip,72554591,1771226178000
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023q1/,faers_ascii_2023q1/,0,1771234754073
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023q1.zip,faers_ascii_2023q1.zip,67476029,1771226168000
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023q2/,faers_ascii_2023q2/,0,1771234754073
dbfs:/Volumes/workspace/default/faers/faers_ascii_2023q2.zip,faers_ascii_2023q2.zip,67650543,1771226171000
dbfs:/Volumes/workspace/default/faers/faers_ascii_2024Q4/,faers_ascii_2024Q4/,0,1771234754073
dbfs:/Volumes/workspace/default/faers/faers_ascii_2024Q4.zip,faers_ascii_2024Q4.zip,68830688,1771227285000


In [0]:
import zipfile
import os

volume_path = "/Volumes/workspace/default/faers"

zip_files = [
    "faers_ascii_2023q1.zip", "faers_ascii_2023q2.zip","faers_ascii_2023Q3.zip", "faers_ascii_2023Q4.zip", "faers_ascii_2024q1.zip","faers_ascii_2024q2.zip", "faers_ascii_2024q3.zip",  "faers_ascii_2024Q4.zip", "faers_ascii_2025q1.zip", "faers_ascii_2025q2 (4).zip","faers_ascii_2025q3 (2).zip", "faers_ascii_2025Q4.zip",
]

for z in zip_files:
    zip_path = f"{volume_path}/{z}"
    extract_path = f"{volume_path}/{z.replace('.zip','')}"
    
    os.makedirs(extract_path, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Extraction completed")


Extraction completed


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, lit

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Your drug list
drug_list = [
    "Lamotrigine", "Levetiracetam", "Topiramate", "Gabapentin", "Pregabalin",
    "Oxcarbazepine", "Zonisamide", "Lacosamide", "Clobazam", "Phenytoin",
    "Carbamazepine", "Phenobarbital", "Valproic acid", "Sodium valproate",
    "Ethosuximide", "Levodopa + Carbidopa", "Bromocriptine", "Pramipexole",
    "Ropinirole", "Rotigotine", "Apomorphine", "Selegiline", "Rasagiline",
    "Safinamide", "Entacapone", "Tolcapone", "Trihexyphenidyl", "Benzhexol",
    "Benztropine", "Amantadine", "Donepezil", "Rivastigmine", "Galantamine",
    "Memantine", "Interferon beta-1a", "Interferon beta-1b", "Glatiramer acetate",
    "Fingolimod", "Teriflunomide", "Dimethyl fumarate", "Natalizumab",
    "Ocrelizumab", "Alemtuzumab", "Baclofen", "Tizanidine", "Modafinil",
    "Sumatriptan", "Rizatriptan", "Zolmitriptan", "Ibuprofen", "Naproxen",
    "Ergotamine", "Dihydroergotamine", "Metoclopramide", "Domperidone",
    "Propranolol", "Amitriptyline", "Candesartan", "Botulinum toxin A",
    "Nortriptyline", "Duloxetine", "Tetrabenazine", "Deutetrabenazine",
    "Haloperidol", "Risperidone", "Diazepam", "Pyridostigmine", "Neostigmine",
    "Prednisolone", "Azathioprine", "Mycophenolate mofetil", "Cyclosporine",
    "Eculizumab", "Rituximab", "Plasmapheresis", "IV immunoglobulin",
    "Zolpidem", "Zopiclone", "Melatonin", "Sodium oxybate", "Methylphenidate",
    "Ceftriaxone", "Vancomycin", "Acyclovir", "Amphotericin B", "Citicoline",
    "Piracetam", "Cerebrolysin", "Edaravone"
]

# Convert drug list to uppercase for matching
drug_list_upper = [d.upper() for d in drug_list]

# Map each quarter folder to its correct DRUG file name
quarter_files = {
    "faers_ascii_2023q1": "DRUG23Q1.txt",
    "faers_ascii_2023q2": "DRUG23Q2.txt",
    "faers_ascii_2023Q3": "DRUG23Q3.txt",
    "faers_ascii_2023Q4": "DRUG23Q4.txt",
    "faers_ascii_2024q1": "DRUG24Q1.txt",
    "faers_ascii_2024q2": "DRUG24Q2.txt",  
    "faers_ascii_2024q3": "DRUG24Q3.txt",
    "faers_ascii_2024Q4": "DRUG24Q4.txt",
    "faers_ascii_2025q1": "DRUG25Q1.txt",
    "faers_ascii_2025q2 (4)": "DRUG25Q2.txt",
    "faers_ascii_2025q3 (2)": "DRUG25Q3.txt",
    "faers_ascii_2025Q4": "DRUG25Q4.txt"
}

dfs = []
for q, fname in quarter_files.items():
    path = f"dbfs:/Volumes/workspace/default/faers/{q}/ASCII/{fname}"
    
    # Read the DRUG file for this quarter
    df = spark.read.csv(path, sep="$", header=True, inferSchema=True)

    # Filter by drug list AND role code (PS, SS only) 
    filtered = df.filter( (upper(col("drugname")).isin(drug_list_upper)) & (col("role_cod").isin(["PS", "SS"])) )
    
    dfs.append(filtered)

# Union all quarters together
final_df = dfs[0]
for d in dfs[1:]:
    final_df = final_df.union(d)



In [0]:
# Show results
final_df.show(20, truncate=False)


+----------+--------+--------+--------+-------------+-------------------------+-------+----------------+-----------------------------------------------------+------------+-------------+------+------+-------+------+-------+--------+---------+---------------------+---------+
|primaryid |caseid  |drug_seq|role_cod|drugname     |prod_ai                  |val_vbm|route           |dose_vbm                                             |cum_dose_chr|cum_dose_unit|dechal|rechal|lot_num|exp_dt|nda_num|dose_amt|dose_unit|dose_form            |dose_freq|
+----------+--------+--------+--------+-------------+-------------------------+-------+----------------+-----------------------------------------------------+------------+-------------+------+------+-------+------+-------+--------+---------+---------------------+---------+
|100264472 |10026447|1       |PS      |RITUXIMAB    |RITUXIMAB                |1      |Intravenous drip|twice within two weeks                               |NULL        |NULL   

In [0]:
final_df.columns


['primaryid',
 'caseid',
 'drug_seq',
 'role_cod',
 'drugname',
 'prod_ai',
 'val_vbm',
 'route',
 'dose_vbm',
 'cum_dose_chr',
 'cum_dose_unit',
 'dechal',
 'rechal',
 'lot_num',
 'exp_dt',
 'nda_num',
 'dose_amt',
 'dose_unit',
 'dose_form',
 'dose_freq']

In [0]:
from pyspark.sql.functions import col, when

# 1. Keep only the required columns
df_selected = final_df.select("caseid", "drugname", "dose_freq", "prod_ai")

# 2. Remove duplicates where (caseid, drugname) are the same
df_dedup = df_selected.dropDuplicates(["caseid", "drugname"])

# 3. Handle unusual / outlier dose_freq values
valid_freqs = [ "QD","BID","TID","QID","QHS","PRN","QOD","QWK","QMO", "Q2H","Q4H","Q6H","Q8H","Q12H","Q24H","STAT","ONCE" ]

df_filtered = df_dedup.filter(col("dose_freq").isin(valid_freqs))

# 4. Convert "UNKNOWN" values to NaN (null in Spark)
df_clean = df_filtered.withColumn(
    "dose_freq",
    when(col("dose_freq") == "UNKNOWN", None).otherwise(col("dose_freq"))
).withColumn(
    "prod_ai",
    when(col("prod_ai") == "UNKNOWN", None).otherwise(col("prod_ai"))
).withColumn(
    "drugname",
    when(col("drugname") == "UNKNOWN", None).otherwise(col("drugname"))
)

# Show results
df_clean.show(20, truncate=False)


+--------+---------------------+---------+---------------------+
|caseid  |drugname             |dose_freq|prod_ai              |
+--------+---------------------+---------+---------------------+
|17924987|PREDNISOLONE         |QD       |PREDNISOLONE         |
|14472611|METOCLOPRAMIDE       |QID      |METOCLOPRAMIDE       |
|13743920|MYCOPHENOLATE MOFETIL|BID      |MYCOPHENOLATE MOFETIL|
|17453003|IBUPROFEN            |BID      |IBUPROFEN            |
|18282701|TERIFLUNOMIDE        |QD       |TERIFLUNOMIDE        |
|14579487|PREDNISOLONE         |QD       |PREDNISOLONE         |
|17280132|NAPROXEN             |QD       |NAPROXEN             |
|18077366|DIAZEPAM             |QD       |DIAZEPAM             |
|17911756|LAMOTRIGINE          |QD       |LAMOTRIGINE          |
|18719386|GABAPENTIN           |QD       |GABAPENTIN           |
|15995631|MYCOPHENOLATE MOFETIL|BID      |MYCOPHENOLATE MOFETIL|
|18789151|CANDESARTAN          |QD       |CANDESARTAN          |
|18948008|CARBAMAZEPINE  

In [0]:
# Example: load all REAC, DEMO, OUTCOME tables across quarters

react_dfs = []
demo_dfs = []
outcome_dfs = []

for q, fname in quarter_files.items():
    try:
        react = spark.read.csv(f"dbfs:/Volumes/workspace/default/faers/{q}/ASCII/REAC{fname[4:]}", 
                               sep="$", header=True, inferSchema=True)
        demo  = spark.read.csv(f"dbfs:/Volumes/workspace/default/faers/{q}/ASCII/DEMO{fname[4:]}", 
                               sep="$", header=True, inferSchema=True)
        outc  = spark.read.csv(f"dbfs:/Volumes/workspace/default/faers/{q}/ASCII/OUTC{fname[4:]}", 
                               sep="$", header=True, inferSchema=True)

        react_dfs.append(react)
        demo_dfs.append(demo)
        outcome_dfs.append(outc)

    except Exception as e:
        print(f"Skipping {q}: {e}")

# Union all quarters for each table
react_all   = react_dfs[0]
for r in react_dfs[1:]:
    react_all = react_all.union(r)

demo_all    = demo_dfs[0]
for d in demo_dfs[1:]:
    demo_all = demo_all.union(d)

outcome_all = outcome_dfs[0]
for o in outcome_dfs[1:]:
    outcome_all = outcome_all.union(o)


In [0]:
# Now merge them into final_df using caseid
final_with_all = (df_clean
                  .join(react_all,   on="caseid", how="left")
                  .join(demo_all,    on="caseid", how="left")
                  .join(outcome_all, on="caseid", how="left"))

# Show merged results
final_with_all.show(20, truncate=False)

+--------+----------+---------+----------+----------+-----------------------------------------+------------+----------+-----------+--------+--------+--------+-----------+--------+--------+--------+------------------------+--------+-------+---+-------+-------+---+-----+----+------+--------+------+--------+----------------+------------+----------+--------+
|caseid  |drugname  |dose_freq|prod_ai   |primaryid |pt                                       |drug_rec_act|primaryid |caseversion|i_f_code|event_dt|mfr_dt  |init_fda_dt|fda_dt  |rept_cod|auth_num|mfr_num                 |mfr_sndr|lit_ref|age|age_cod|age_grp|sex|e_sub|wt  |wt_cod|rept_dt |to_mfr|occp_cod|reporter_country|occr_country|primaryid |outc_cod|
+--------+----------+---------+----------+----------+-----------------------------------------+------------+----------+-----------+--------+--------+--------+-----------+--------+--------+--------+------------------------+--------+-------+---+-------+-------+---+-----+----+------+-----

In [0]:
# Select only the required columns from final_with_all
master_df = final_with_all.select(
    "caseid",
    "drugname",
    "dose_freq",
    "prod_ai",
    "pt",              # reaction term
    "drug_rec_act",    # drug action taken
    "age_grp",         # age group
    "sex",             # patient sex
    "outc_cod"         # outcome code
)

# Show results
master_df.show(20, truncate=False)


+--------+------------+---------+------------+-------------------------------------------------+------------+-------+----+--------+
|caseid  |drugname    |dose_freq|prod_ai     |pt                                               |drug_rec_act|age_grp|sex |outc_cod|
+--------+------------+---------+------------+-------------------------------------------------+------------+-------+----+--------+
|13413479|RISPERIDONE |QD       |RISPERIDONE |Amnesia                                          |NULL        |NULL   |M   |OT      |
|13413479|RISPERIDONE |QD       |RISPERIDONE |Initial insomnia                                 |NULL        |NULL   |M   |OT      |
|13413479|RISPERIDONE |QD       |RISPERIDONE |Increased appetite                               |NULL        |NULL   |M   |OT      |
|13413479|RISPERIDONE |QD       |RISPERIDONE |Somnolence                                       |NULL        |NULL   |M   |OT      |
|13413479|RISPERIDONE |QD       |RISPERIDONE |Emotional distress            

In [0]:
master_df.columns

['caseid',
 'drugname',
 'dose_freq',
 'prod_ai',
 'pt',
 'drug_rec_act',
 'age_grp',
 'sex',
 'outc_cod']

In [0]:
from pyspark.sql.functions import col, count, when

# 1. Remove duplicates by caseid + drugname
master_df = master_df.dropDuplicates(["caseid", "drugname"])

# 2. Drop columns with >60% null values
row_count = master_df.count()

# Compute null counts for each column
null_counts = master_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in master_df.columns
]).collect()[0].asDict()

# Keep only columns where null percentage <= 60%
valid_cols = [c for c in master_df.columns if (null_counts[c] / row_count) <= 0.6]

# Re‑select only valid columns
master_df = master_df.select(*valid_cols)

# Show cleaned master_df
master_df.show(20, truncate=False)


+--------+---------------------+---------+---------------------+--------------------------------------------+----+--------+
|caseid  |drugname             |dose_freq|prod_ai              |pt                                          |sex |outc_cod|
+--------+---------------------+---------+---------------------+--------------------------------------------+----+--------+
|17770939|PREDNISOLONE         |QD       |PREDNISOLONE         |Renal impairment                            |M   |HO      |
|16561171|PREDNISOLONE         |QD       |PREDNISOLONE         |Pneumocystis jirovecii pneumonia            |M   |OT      |
|19289993|LACOSAMIDE           |BID      |LACOSAMIDE           |Petit mal epilepsy                          |M   |DS      |
|13413479|RISPERIDONE          |QD       |RISPERIDONE          |Amnesia                                     |M   |OT      |
|18853661|CEFTRIAXONE          |QD       |CEFTRIAXONE          |Hyponatraemia                               |M   |OT      |
|1919475

In [0]:
# Find rows where drugname and prod_ai differ
diff_df = master_df.filter(col("drugname") != col("prod_ai")) 
# Count how many such rows exist
diff_count = diff_df.count() 
print(f"Number of rows where drugname != prod_ai: {diff_count}")

Number of rows where drugname != prod_ai: 1589


In [0]:
# Show a few examples 
diff_df.select("caseid", "drugname", "prod_ai").show(20, truncate=False)

+--------+-----------+---------------------------------------+
|caseid  |drugname   |prod_ai                                |
+--------+-----------+---------------------------------------+
|23514053|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM         |
|23509878|ZOLPIDEM   |ZOLPIDEM TARTRATE                      |
|23993144|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM         |
|23521714|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM         |
|24074815|PROPRANOLOL|PROPRANOLOL HYDROCHLORIDE              |
|23754751|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM         |
|16363322|PRAMIPEXOLE|PRAMIPEXOLE\PRAMIPEXOLE DIHYDROCHLORIDE|
|23482899|ZOLPIDEM   |ZOLPIDEM TARTRATE                      |
|24103344|PROPRANOLOL|PROPRANOLOL HYDROCHLORIDE              |
|23729016|ZOLPIDEM   |ZOLPIDEM TARTRATE                      |
|25023968|ZOLPIDEM   |ZOLPIDEM TARTRATE                      |
|23155125|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM         |
|23513121|CEFTRIAXONE|CEFTRIAXONE\CEFTRIAXONE SODIUM   

In [0]:
# Number of rows
row_count = master_df.count()

# Number of columns
col_count = len(master_df.columns)

print(f"Master DF shape: ({row_count}, {col_count})")


Master DF shape: (54861, 7)


In [0]:
for q, files in quarter_files.items():
    df = spark.read.csv(f"dbfs:/Volumes/workspace/default/faers/{q}/ASCII/{files}", sep="$", header=True, inferSchema=True)
    filtered = df.filter(
        (upper(col("drugname")).isin(drug_list_upper)) &
        (col("role_cod").isin(["PS","SS"]))
    )
    print(q, filtered.count())


faers_ascii_2023q1 52261
faers_ascii_2023q2 53950
faers_ascii_2023Q3 50072
faers_ascii_2023Q4 55030
faers_ascii_2024q1 56031
faers_ascii_2024q2 61103
faers_ascii_2024q3 60113
faers_ascii_2024Q4 68663
faers_ascii_2025q1 62319
faers_ascii_2025q2 (4) 56277
faers_ascii_2025q3 (2) 74709
faers_ascii_2025Q4 59142


In [0]:
master_df.write.format("delta").mode("overwrite").save("dbfs:/Volumes/workspace/default/faers/master_df")
